In [ ]:
%pip install pandas
import pandas as pd

products = pd.DataFrame(columns=[
    "product_id",
    "product_name",
    "sku_upc",
    "product_type",
    "retailer",
    "msrp",
    "exclusive_flag",
    "release_wave",
    "notes"
])

products

,product_id,product_name,sku_upc,product_type,retailer,msrp,exclusive_flag,release_wave,notes


In [ ]:
products.loc[len(products)] = [
    2,
    "Paldea Evolved Booster Box",
    "987654321",
    "Booster Box",
    "Target",
    161.64,
    False,
    "SV Paldea Evolved",
    "Standard booster box"
]

products

In [14]:
assert products["product_id"].is_unique, "Duplicate product_id detected"
assert products["product_name"].notnull().all(), "Missing product names"

In [15]:
import os

# Ensure data directory exists
os.makedirs("../data", exist_ok=True)

# Save product master
products.to_csv("../data/products.csv", index=False)

print("✅ products.csv created in /data")

✅ products.csv created in /data


In [16]:
import pandas as pd
from pathlib import Path

# --- Load existing product master ---
products_path = Path("../data/products.csv")
products = pd.read_csv(products_path)

# --- Define sets + product templates (32 total) ---
sets = [
    "Ascended Heroes",
    "Destined Rivals",
    "White Flare",
    "Black Bolt",
    "Surging Sparks",
    "Perfect Order",
    "Phantasmal Flames",
    "Mega Evolution",
]

product_templates = [
    ("ETB", "Elite Trainer Box"),
    ("Pin Collection", "Pin Collection"),
    ("Poster Collection", "Poster Collection"),
    ("Booster Bundle", "Booster Bundle"),
]

# --- Compute next product_id ---
products["product_id"] = pd.to_numeric(products["product_id"], errors="coerce")
start_id = int(products["product_id"].max()) + 1 if products["product_id"].notna().any() else 1

# --- Build new rows ---
new_rows = []
pid = start_id
for s in sets:
    for ptype, display in product_templates:
        new_rows.append({
            "product_id": pid,
            "product_name": f"{s} - {display}",
            "sku_upc": "",                 # fill later if you want
            "product_type": ptype,         # ETB / Pin Collection / Poster Collection / Booster Bundle
            "retailer": "Multi",           # retailer-agnostic catalog
            "msrp": "",                    # fill later (kept blank to avoid wrong assumptions)
            "exclusive_flag": False,
            "release_wave": s,
            "notes": ""
        })
        pid += 1

new_products = pd.DataFrame(new_rows)

# --- Append, de-dupe by product_name (safe reruns), save ---
combined = pd.concat([products, new_products], ignore_index=True)
combined = combined.drop_duplicates(subset=["product_name"], keep="first")

combined.to_csv(products_path, index=False)

print(f"✅ Added {len(new_products)} products (after de-dupe: {len(combined) - len(products)} net new). Saved to {products_path}")
combined.tail(40)


✅ Added 32 products (after de-dupe: 32 net new). Saved to ..\data\products.csv


,product_id,product_name,sku_upc,product_type,retailer,msrp,exclusive_flag,release_wave,notes
0,2,Paldea Evolved Booster Box,987654321,Booster Box,Target,161.64,False,SV Paldea Evolved,Standard booster box
1,3,Ascended Heroes - Elite Trainer Box,,ETB,Multi,,False,Ascended Heroes,
2,4,Ascended Heroes - Pin Collection,,Pin Collection,Multi,,False,Ascended Heroes,
3,5,Ascended Heroes - Poster Collection,,Poster Collection,Multi,,False,Ascended Heroes,
4,6,Ascended Heroes - Booster Bundle,,Booster Bundle,Multi,,False,Ascended Heroes,
5,7,Destined Rivals - Elite Trainer Box,,ETB,Multi,,False,Destined Rivals,
6,8,Destined Rivals - Pin Collection,,Pin Collection,Multi,,False,Destined Rivals,
7,9,Destined Rivals - Poster Collection,,Poster Collection,Multi,,False,Destined Rivals,
8,10,Destined Rivals - Booster Bundle,,Booster Bundle,Multi,,False,Destined Rivals,
9,11,White Flare - Elite Trainer Box,,ETB,Multi,,False,White Flare,


In [17]:
import pandas as pd
from pathlib import Path

products_path = Path("../data/products.csv")
products = pd.read_csv(products_path)

MSRP_BY_TYPE = {
    "ETB": 59.99,
    "Booster Bundle": 26.94,
    "Pin Collection": 29.99,
    "Poster Collection": 49.99,
}

products["msrp"] = products.apply(
    lambda row: MSRP_BY_TYPE.get(row["product_type"], row["msrp"]),
    axis=1
)

products.to_csv(products_path, index=False)

products[
    ["product_id", "product_name", "product_type", "msrp"]
].sort_values("product_id").tail(40)

,product_id,product_name,product_type,msrp
0,2,Paldea Evolved Booster Box,Booster Box,161.64
1,3,Ascended Heroes - Elite Trainer Box,ETB,59.99
2,4,Ascended Heroes - Pin Collection,Pin Collection,29.99
3,5,Ascended Heroes - Poster Collection,Poster Collection,49.99
4,6,Ascended Heroes - Booster Bundle,Booster Bundle,26.94
5,7,Destined Rivals - Elite Trainer Box,ETB,59.99
6,8,Destined Rivals - Pin Collection,Pin Collection,29.99
7,9,Destined Rivals - Poster Collection,Poster Collection,49.99
8,10,Destined Rivals - Booster Bundle,Booster Bundle,26.94
9,11,White Flare - Elite Trainer Box,ETB,59.99
